In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
!pip install pyreadstat
import pyreadstat #since the data files are .xpt files, this library is needed to import the table
import re
import requests
from bs4 import BeautifulSoup
from nhanes_utils import to_snake_case, get_common_nan_ids, standardize_id_column, drop_rows_with_common_nan_ids

In [2]:
file_path = '2017-2020/P_PAQ.xpt'

df, meta = pyreadstat.read_xport(file_path)
df = standardize_id_column(df)

In [3]:
df.head()

,participant_id,PAQ605,PAQ610,PAD615,PAQ620,PAQ625,PAD630,PAQ635,PAQ640,PAD645,PAQ650,PAQ655,PAD660,PAQ665,PAQ670,PAD675,PAD680
0,109266.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,5.0,60.0,1.0,4.0,30.0,480.0
1,109267.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,6.0,90.0,2.0,NaN,NaN,540.0
2,109268.0,1.0,5.0,540.0,1.0,5.0,300.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,540.0
3,109271.0,2.0,NaN,NaN,1.0,2.0,120.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,60.0
4,109273.0,1.0,3.0,240.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,4.0,120.0,180.0


In [6]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PAQ.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

pattern = re.compile(r'^([A-Z0-9_]+)\s*-\s*(.+)$')

rename_dict = {}

# Loop through all h3 tags and filter those that match our pattern
for tag in soup.find_all('h3'):
    text = tag.get_text(strip=True)
    match = pattern.match(text)
    if match:
        var_name = match.group(1)
        description = match.group(2)
                
        clean_name = to_snake_case(description)
        rename_dict[var_name] = clean_name

In [7]:
filtered_rename_dict = {
    k: v for k, v in rename_dict.items() if k in df.columns
}

df = df.rename(columns=filtered_rename_dict)

In [8]:
df.head()

,participant_id,vigorous_work_activity,number_of_days_vigorous_work,minutes_vigorous_intensity_work,moderate_work_activity,number_of_days_moderate_work,minutes_moderate_intensity_work,walk_or_bicycle,number_of_days_walk_or_bicycle,minutes_walk_bicycle_for_transportation,vigorous_recreational_activities,days_vigorous_recreational_activities,minutes_vigorous_recreational_activities,moderate_recreational_activities,days_moderate_recreational_activities,minutes_moderate_recreational_activities,minutes_sedentary_activity
0,109266.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,5.0,60.0,1.0,4.0,30.0,480.0
1,109267.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,6.0,90.0,2.0,NaN,NaN,540.0
2,109268.0,1.0,5.0,540.0,1.0,5.0,300.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,540.0
3,109271.0,2.0,NaN,NaN,1.0,2.0,120.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,60.0
4,109273.0,1.0,3.0,240.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,4.0,120.0,180.0


In [9]:
df.isna().sum()

participant_id                                 0
vigorous_work_activity                         0
number_of_days_vigorous_work                7275
minutes_vigorous_intensity_work             7291
moderate_work_activity                         0
number_of_days_moderate_work                5506
minutes_moderate_intensity_work             5529
walk_or_bicycle                                0
number_of_days_walk_or_bicycle              7440
minutes_walk_bicycle_for_transportation     7452
vigorous_recreational_activities               0
days_vigorous_recreational_activities       7271
minutes_vigorous_recreational_activities    7276
moderate_recreational_activities               0
days_moderate_recreational_activities       5789
minutes_moderate_recreational_activities    5800
minutes_sedentary_activity                    17
dtype: int64

In [10]:
df.shape

(9693, 17)